# F6-svd-spectral — Session 1: Eigenvalues and Eigenvectors

*One class session, roughly 85 minutes. Prerequisites: F3-matrices
(matrix action, rank, linear independence, Gram matrices) and, through
it, F2-vectors (dot products, norms, unit vectors, orthonormality).*

**This session:** the special directions a matrix does not turn
(eigenvectors) and the stretch factors it applies along them
(eigenvalues); how to find both *by hand* for $2 \times 2$ matrices
using nothing but F3's linear-dependence vocabulary and the quadratic
formula; the library call `np.linalg.eig` and how to read its output
safely; what eigenvalue signs and magnitudes do under repeated
application; and the special promises symmetric matrices make.
Plus a fully worked exam-style multiple-choice problem in the
normal-form register.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np

SEED = 20260804

## 1. A Direction the Map Does Not Turn

**Motivation.**
F3 taught you to read a matrix as a machine acting on vectors.
Feed the machine a direction and it generally does two things at once:
it **rotates** the direction and it **stretches** it.
This unit begins with a simple question that turns out to organize
everything from here to the end of the course:

> Are there input directions the machine does *not* rotate — directions
> it only stretches?

Take the symmetric matrix

$$A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}.$$

Feed it a fan of unit directions and, for each input $x$, compare the
direction of $Ax$ with the direction of $x$.
F2 gave us the exact tool for "same direction or not": the **cosine**
of the angle between the two vectors,
$\cos\theta = \dfrac{x \cdot Ax}{\lVert x \rVert\,\lVert Ax \rVert}$.
Cosine $+1$ means $Ax$ points exactly along $x$ (unturned), $-1$ means
exactly opposite, anything else means the machine rotated the input.

In [ ]:
A = np.array([[2., 1.], [1., 2.]])

angles = np.deg2rad(np.arange(0, 180, 22.5))
print("input angle | cos(x, Ax)  | stretch |Ax|/|x|")
for t in angles:
    x = np.array([np.cos(t), np.sin(t)])
    y = A @ x
    cos = (x * y).sum() / (np.sqrt((x**2).sum()) * np.sqrt((y**2).sum()))
    stretch = np.sqrt((y**2).sum()) / np.sqrt((x**2).sum())
    print(f"  {np.rad2deg(t):6.1f}    |  {cos:+.6f}  |  {stretch:.6f}")

Read the table: most directions come out with $\cos(x, Ax) < 1$ —
rotated.
But two inputs stand out with cosine exactly $+1.000000$:

- the $45°$ direction $(1, 1)/\sqrt2$, which comes out **unturned and
  stretched by $3$**, and
- the $135°$ direction $(-1, 1)/\sqrt2$, which comes out unturned and
  stretched by $1$ (unchanged).

Those two directions are this matrix's **eigenvectors**, and the
stretch factors $3$ and $1$ are its **eigenvalues**.
The rest of this session makes that precise, teaches you to find both
by hand and by library call, and starts collecting the facts the whole
unit builds on.

### Checkpoint 1

1. Compute $A\,(1, 1)$ and $A\,(-1, 1)$ by hand
   (row·input dot products, F3 style) and confirm the stretch factors
   $3$ and $1$ read off the table.
2. The table probed the $45°$ direction with a *unit* vector.
   Is $(2, 2)$ — same direction, length $2\sqrt2$ — also unturned by
   $A$?
   Compute $A\,(2,2)$ and reason in one sentence.
3. From the table alone, is the $0°$ direction $(1, 0)$ an eigenvector
   of $A$?
   Which number in its row tells you?

## 2. The Eigen Equation $S q = \lambda q$

**Definition.**
Let $S$ be a square $(n, n)$ matrix.
A **nonzero** vector $q$ is an **eigenvector** of $S$ when $S$ maps it
to a multiple of itself:

$$S q = \lambda q$$

for some number $\lambda$, called the **eigenvalue** belonging to $q$.
We write eigenpairs $(\lambda, q)$.
Geometrically: the direction of $q$ is preserved (or exactly reversed,
if $\lambda < 0$), and $\lambda$ is the *signed* stretch factor.

Two immediate consequences of the definition:

- **Scaling freedom.** If $q$ is an eigenvector with eigenvalue
  $\lambda$, so is every nonzero multiple $cq$:
  $S(cq) = c\,Sq = c\lambda q = \lambda\,(cq)$.
  An eigenvector is really an eigen-*direction*; any nonzero vector
  along it represents it.
  (This freedom is why different routes to "the" eigenvector can
  disagree by a factor — most often a sign.
  Session 2 pins the course convention for comparing them.)
- **Why $q = 0$ is excluded.** $S\,0 = \lambda\,0$ holds for *every*
  $\lambda$ — it certifies nothing.
  Only nonzero vectors carry direction information, so only they count.

**Worked check.**
Take $S = \begin{pmatrix} 4 & 2 \\ 1 & 3 \end{pmatrix}$ and test three
candidates:

- $q = (2, 1)$: $\;Sq = (4\cdot2 + 2\cdot1,\; 1\cdot2 + 3\cdot1)
  = (10, 5) = 5\,(2, 1)$. ✓ eigenvector, $\lambda = 5$.
- $q = (1, -1)$: $\;Sq = (4 - 2,\; 1 - 3) = (2, -2) = 2\,(1, -1)$.
  ✓ eigenvector, $\lambda = 2$.
- $q = (1, 0)$: $\;Sq = (4, 1)$ — not a multiple of $(1, 0)$
  (a second entry appeared from nowhere). ✗ not an eigenvector.

In [ ]:
S = np.array([[4., 2.], [1., 3.]])

for q in [np.array([2., 1.]), np.array([1., -1.]), np.array([1., 0.])]:
    y = S @ q
    # multiple-of test, F3 style: y and q proportional <=> cross-product zero
    cross = y[0] * q[1] - y[1] * q[0]
    print(f"q = {q}  Sq = {y}  cross = {cross:+.1f}",
          "-> eigenvector" if cross == 0 else "-> NOT an eigenvector")

The code uses F3's proportionality test: $y$ is a multiple of $q$
exactly when the cross-product $y_0 q_1 - y_1 q_0$ vanishes.
Keep that test in your kit — it returns in the very next section
wearing a bigger hat.

### Checkpoint 2

1. For the same $S$, test the candidates $(4, 2)$ and $(1, 1)$ by hand:
   eigenvector or not, and if so with what eigenvalue?
2. Show in one line that if $(\lambda, q)$ is an eigenpair of $S$, then
   $(\lambda, -q)$ is too.
3. A classmate says "$q = (0, 0)$ satisfies $Sq = 7q$, so $7$ is an
   eigenvalue of every matrix."
   What exactly is wrong with that?

## 3. Finding Eigenvalues by Hand: the Dependent-Rows Route

`np.linalg.eig` will do this numerically in Section 5 — but the exam's
hand-register items, and every ounce of your intuition, come from being
able to do $2 \times 2$ cases on paper.
The route below uses only tools you already own from F3.

**The logic chain.**
Fix a candidate number $\lambda$ and ask: does
$S = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$
have an eigenvector with this eigenvalue?

$$
S q = \lambda q
\;\iff\; S q - \lambda q = 0
\;\iff\; (S - \lambda I)\, q = 0
\quad\text{for some } q \ne 0.
$$

So $\lambda$ is an eigenvalue exactly when the matrix
$S - \lambda I = \begin{pmatrix} a - \lambda & b \\ c & d - \lambda
\end{pmatrix}$
sends some *nonzero* vector to zero.
F3 told you precisely when that happens: a square matrix collapses a
nonzero input to $0$ exactly when it is **not full rank** — when its
rows are **linearly dependent**.
For a $2 \times 2$ matrix, dependent rows means one row is a multiple
of the other: the rows $(a - \lambda,\; b)$ and $(c,\; d - \lambda)$
are **proportional**.
And proportionality of two 2-vectors is Section 2's cross-product test:

$$(a - \lambda)(d - \lambda) - b\,c = 0.$$

That is a **quadratic in $\lambda$** — Calc AB algebra from here.
Its roots are the eigenvalues.

**Worked example.**
$S = \begin{pmatrix} 4 & 2 \\ 1 & 3 \end{pmatrix}$:

$$(4 - \lambda)(3 - \lambda) - 2 \cdot 1
= \lambda^2 - 7\lambda + 10 = (\lambda - 5)(\lambda - 2) = 0
\;\Rightarrow\; \lambda = 5 \text{ or } 2$$

— exactly the stretch factors we verified in Section 2.
When the quadratic does not factor by eye, the quadratic formula
finishes the job.

> **Vocabulary aside (one line).** The combination $ad - bc$ is called
> the **determinant** of a $2\times2$ matrix; we need only the name —
> this course never uses determinants beyond this dependent-rows test.

**Convention (course-wide pin).** Eigenvalues are always listed in
**descending** order: $\lambda_1 \ge \lambda_2 \ge \cdots$

In [ ]:
# Both roots really do make the rows of S - lam*I proportional:
a, b, c, d = 4., 2., 1., 3.
for lam in [5.0, 2.0]:
    cross = (a - lam) * (d - lam) - b * c
    print(f"lambda = {lam}: (a-l)(d-l) - bc = {cross:+.1f}",
          "-> rows dependent, eigenvalue confirmed" if cross == 0 else "")

### Checkpoint 3

1. Find both eigenvalues of
   $\begin{pmatrix} 3 & 1 \\ 2 & 2 \end{pmatrix}$ by the dependent-rows
   route (set up the quadratic, factor or use the formula).
   List them in descending order.
2. Find both eigenvalues of
   $\begin{pmatrix} 2 & 3 \\ 0 & 5 \end{pmatrix}$.
   What made this one especially fast?
3. The chain above required "$(S - \lambda I)q = 0$ for some
   $q \ne 0$."
   If we dropped the "$q \ne 0$" condition, which step of the chain
   breaks, and what silly conclusion would follow?

## 4. From $\lambda$ Back to $q$

Knowing an eigenvalue, the matching eigenvector is one proportionality
read away.
$(S - \lambda I)\,q = 0$ says: **each row of $S - \lambda I$ has dot
product $0$ with $q$** — and since the rows are proportional (that is
what made $\lambda$ an eigenvalue), one row's worth of information is
all there is.

**Recipe ($2 \times 2$).** Take any *nonzero* row $(r_0, r_1)$ of
$S - \lambda I$; then

$$q = (r_1, \; -r_0)$$

works, because $(r_0, r_1) \cdot (r_1, -r_0) = r_0 r_1 - r_1 r_0 = 0$.
(Any nonzero multiple of that $q$ is equally correct — scaling
freedom.)

**Worked example**, continuing
$S = \begin{pmatrix} 4 & 2 \\ 1 & 3 \end{pmatrix}$:

- $\lambda = 5$: $\;S - 5I = \begin{pmatrix} -1 & 2 \\ 1 & -2
  \end{pmatrix}$; top row $(-1, 2)$ gives $q = (2, 1)$.
  Check: $S\,(2,1) = (10, 5) = 5\,(2,1)$. ✓
- $\lambda = 2$: $\;S - 2I = \begin{pmatrix} 2 & 2 \\ 1 & 1
  \end{pmatrix}$; top row $(2, 2)$ gives $q = (2, -2)$, or after
  rescaling $q = (1, -1)$.
  Check: $S\,(1,-1) = (2, -2) = 2\,(1,-1)$. ✓

In [ ]:
S = np.array([[4., 2.], [1., 3.]])

for lam in [5.0, 2.0]:
    M = S - lam * np.eye(2)
    r0, r1 = M[0]            # top row of S - lam*I
    q = np.array([r1, -r0])  # perpendicular-read recipe
    resid = np.abs(S @ q - lam * q).max()
    print(f"lambda = {lam}: q = {q}, max|Sq - lam q| = {resid:.1f}")

### Checkpoint 4

1. Continue Checkpoint 3's matrix
   $\begin{pmatrix} 3 & 1 \\ 2 & 2 \end{pmatrix}$: find an eigenvector
   for *each* of its two eigenvalues, and verify both by a hand
   multiplication.
2. In the recipe, why must you pick a *nonzero* row of
   $S - \lambda I$?
   And why is picking the bottom row (when nonzero) guaranteed to give
   the same *direction* as the top row?

## 5. `np.linalg.eig`

For anything bigger than $2 \times 2$ — and to check hand work — NumPy
supplies the general eigen-solver.
(Earlier units banned `np.linalg` calls in specific problems to force
hand routes; this unit *teaches* the calls themselves — Session 3
states exactly how legality is scoped from here on.)

```python
vals, vecs = np.linalg.eig(S)
```

Three things you must know about its output, each a classic
first-encounter surprise:

1. **`vecs` holds eigenvectors as COLUMNS**: `vecs[:, i]` pairs with
   `vals[i]`.
   Rows of `vecs` mean nothing.
2. **The order of `vals` is arbitrary** — not sorted.
   Our course convention is descending, so reorder every time:
   `order = np.argsort(vals)[::-1]` (the F1 argsort idiom), then
   `vals[order]` and `vecs[:, order]`.
3. **The dtype may be complex.** A general square matrix can have
   complex eigenvalues (a pure rotation turns *every* real direction —
   more below), so `eig` reports complex numbers even when the
   imaginary parts are all zero.
   For the matrices in this course, check the imaginary parts vanish,
   then keep the real parts.

Each returned column is unit length; by scaling freedom, that is just
`eig`'s representative of the eigen-direction — your hand answer
$(2, 1)$ and the returned $(0.894, 0.447) = (2,1)/\sqrt5$ are the same
eigenvector.

In [ ]:
S = np.array([[4., 2.], [1., 3.]])
vals, vecs = np.linalg.eig(S)
print("raw vals:", vals)               # note the dtype
print("raw vecs (columns!):\n", vecs)

imag_max = np.abs(vals.imag).max()
print("max |imag part|:", imag_max, " -> keep the real parts")
vals, vecs = vals.real, vecs.real

order = np.argsort(vals)[::-1]         # descending, the course pin
vals, vecs = vals[order], vecs[:, order]
print("vals (desc):", vals)

for i in range(2):
    resid = np.abs(S @ vecs[:, i] - vals[i] * vecs[:, i]).max()
    print(f"pair {i}: lambda = {vals[i]:.1f}, "
          f"q = {vecs[:, i]}, max|Sq - lam q| = {resid:.2e}")

print("hand q for lambda=5, unit length:",
      np.array([2., 1.]) / np.sqrt(5.))

The check `max|Sq - lam q|` at machine-precision size
($\sim 10^{-16}$) is the **eigen-equation residual** — the
verification idiom this unit uses everywhere, including the capstone.
Make it a reflex: after *any* eigen-computation, by any route, feed
each claimed pair back through the definition.

### Checkpoint 5

1. Run `np.linalg.eig` (mentally or in NumPy) on Checkpoint 3's matrix
   $\begin{pmatrix} 3 & 1 \\ 2 & 2 \end{pmatrix}$ and reconcile the
   output with your hand eigenvalues and eigenvectors — same
   directions?
   Which two output conventions did you have to handle to compare?
2. A script does `vals, vecs = np.linalg.eig(S)` and then treats
   `vecs[0]` as the first eigenvector.
   Exactly what did it grab instead, and what check from this section
   would have exposed the bug immediately?

## 6. Signs and Magnitudes: Eigenvalues as Stretch Factors

An eigenvalue is a *signed* stretch factor along its own direction.
Reading the number tells you what repeated application of the machine
will do there:

| $\lambda$ | one application | applied $k$ times |
|---|---|---|
| $\lambda > 1$ | stretch | grows like $\lambda^k$ |
| $\lambda = 1$ | unchanged | unchanged |
| $0 < \lambda < 1$ | compress | shrinks toward $0$ |
| $\lambda = 0$ | flattened to $0$ | stays $0$ |
| $-1 < \lambda < 0$ | flip and compress | alternates sign, shrinks |
| $\lambda < -1$ | flip and stretch | alternates sign, grows |

Two consequences worth pinning:

- **$\lambda = 0$ means singular.** $Sq = 0$ for a nonzero $q$ is
  exactly F3's "collision": the map is not full rank, its rows are
  dependent, no undo exists.
  "Has $0$ as an eigenvalue" and "rank-deficient" are the same fact in
  two vocabularies.
- **The largest $|\lambda|$ dominates repetition.** Write a generic
  input as a mix of eigen-directions; each application multiplies each
  component by its own $\lambda$; after many applications the direction
  with the largest $|\lambda|$ has outgrown the rest.
  Iterating $x \mapsto Sx$ (renormalizing as you go) therefore *turns
  any generic start vector toward the top eigen-direction* — watch:

In [ ]:
S = np.array([[4., 2.], [1., 3.]])   # eigenvalues 5 and 2

for k in [1, 2, 3, 6, 12]:
    y = np.array([1., 0.])           # NOT an eigenvector (Section 2)
    for _ in range(k):
        y = S @ y
        y = y / np.abs(y).max()      # renormalize: keep entries readable
    print(f"after {k:2d} applications, direction ~ {y}")
print("top eigenvector (2,1), same scaling:", np.array([1., 0.5]))

By twelve applications the iterate is indistinguishable from the
$\lambda = 5$ direction $(2, 1)$ — scaled here so the largest entry is
$1$, i.e. $(1, 0.5)$.
The $\lambda = 2$ component is still present, but it has lost by a
factor of $(2/5)^{12}$.
This *power iteration* picture is the cheapest genuine eigen-algorithm
there is, and one of this unit's challenge problems makes you build it
with `np.linalg` banned entirely.

### Checkpoint 6

1. $D = \begin{pmatrix} 3 & 0 \\ 0 & 0.5 \end{pmatrix}$ is applied
   $10$ times to $x = (1, 1)$.
   Give the exact result (no matrix arithmetic needed — read the
   diagonal as eigenvalues) and describe its direction in words.
2. A $2 \times 2$ matrix has eigenvalues $2$ and $0$.
   What is its rank, and what happens to inputs along the $\lambda = 0$
   eigen-direction?
3. A matrix has eigenvalues $-3$ and $1$.
   Under repeated application, what does a generic input's direction
   settle toward, and what keeps happening to its *sign*?

## 7. Symmetric Matrices Keep Two Promises

Everything so far works for any square matrix — and general square
matrices can misbehave: complex eigenvalues, eigenvectors at awkward
angles to each other.
The matrices this unit actually runs on — Gram matrices
$WW^{\mathsf T}$ from F3, and the $S$ of the SVD bridge in Session 3 —
are all **symmetric** ($S = S^{\mathsf T}$), and symmetric matrices
keep two remarkable promises.

> **Fact (stated, not proved — course register: verified numerically).**
> Every real symmetric matrix $S$ has
> 1. **all-real eigenvalues**, and
> 2. an **orthonormal set of eigenvectors**: unit-length, mutually
>    perpendicular — a full basis of them.
>
> The proof belongs to a later linear-algebra course; *using* the fact
> is this unit's business.

Orthonormality of eigenvectors is a big deal: stack them as the columns
of $Q$ and F2's orthonormality test reads
$Q^{\mathsf T} Q = I$.
Session 2 turns this into the *spectral decomposition* — symmetric
matrices are exactly the machines that stretch space along a set of
perpendicular axes.

In [ ]:
rng = np.random.default_rng(SEED)
B = rng.normal(0, 1, (5, 5))
S5 = B + B.T                    # symmetric by construction
print("symmetry gap:", np.abs(S5 - S5.T).max())

vals, vecs = np.linalg.eig(S5)
print("max |imag part| of eigenvalues:", np.abs(vals.imag).max())
vals, vecs = vals.real, vecs.real
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print("vals (desc):", vals)

# Promise 2: the eigenvector columns are orthonormal (F2 test: QtQ = I)
gap = np.abs(vecs.T @ vecs - np.eye(5)).max()
print("max |Q^T Q - I|:", gap)

Both promises verified on a seeded random symmetric matrix: imaginary
parts exactly zero, and $Q^{\mathsf T}Q = I$ to machine precision.
Note the eigenvalues *themselves* can be negative — promise 1 says
**real**, not positive.
(Which symmetric matrices also promise non-negative eigenvalues is a
Session 2 question with a satisfying answer.)

Contrast with the most famous non-symmetric matrix, the quarter-turn

$$R = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix},$$

which rotates *every* real direction by $90°$ — no real direction
survives unturned, so no real eigenvector can exist, and `eig` answers
with honestly complex eigenvalues $\pm i$:

In [ ]:
R = np.array([[0., -1.], [1., 0.]])
rvals, _ = np.linalg.eig(R)
print("quarter-turn eigenvalues:", rvals)

### Checkpoint 7

1. Why is $Q^{\mathsf T} Q = I$ exactly the statement "the columns of
   $Q$ are unit length and mutually perpendicular"?
   (Say what entry $(i, j)$ of $Q^{\mathsf T} Q$ computes — F3's Gram
   view.)
2. Is the quarter-turn $R$ symmetric?
   Reconcile its complex eigenvalues with the two promises in one
   sentence.
3. $S_5$ above has some negative eigenvalues.
   Does that contradict promise 1?
   What would a negative eigenvalue mean geometrically for its
   eigen-direction?

## 8. Worked Exam-Style Example: Hand Eigenvalues, Normal Form

The real paper's multiple-choice items wrap hand-register skills in a
**numeric normal form**: you compute a number, write it as $\pm m$ with
$m \ge 0$, then decode via a stated rule so exactly one option matches.
(F3 Session 3 §7 introduced this register; here it meets eigenvalues.)

---

**Worked exam-style example 1 (multiple choice, numeric normal form).**

> Let $S = \begin{pmatrix} 7 & 4 \\ 1 & 4 \end{pmatrix}$, with
> eigenvalues listed in descending order $\lambda_1 \ge \lambda_2$.
> Compute $s = \lambda_1 - 2\lambda_2$.
> Your answer can be written as $s = \pm m$ with $m$ a non-negative
> integer.
> What is the value of $2m + 1$ if $s \ge 0$, or $2m$ if $s < 0$?
>
> A. 4  B. 5  C. 6  D. 10  E. 11
>
> Reasoning is not required.

*Solution, step by step.*

1. **Dependent-rows condition.**
   $(7 - \lambda)(4 - \lambda) - 4 \cdot 1 = \lambda^2 - 11\lambda + 24
   = 0.$
2. **Solve.** Factor: $(\lambda - 8)(\lambda - 3) = 0$, so
   $\lambda = 8$ or $3$.
   (Or the quadratic formula:
   $\lambda = \frac{11 \pm \sqrt{121 - 96}}{2} = \frac{11 \pm 5}{2}$.)
3. **Order.** Descending — the pinned convention:
   $\lambda_1 = 8$, $\lambda_2 = 3$.
4. **Target.** $s = 8 - 2 \cdot 3 = 2$.
5. **Normal form.** $s = +2$: $m = 2$, branch $s \ge 0$.
6. **Decode.** $2m + 1 = 5$ → **B**.

Where the distractors come from: drop the $-\lambda$ cross terms
sloppily and the quadratic $\lambda^2 - 10\lambda + 24$ gives
$\lambda = 6, 4$, hence $s = 6 - 8 = -2$, decoding through the *other*
branch to $2m = 4$ — option A, a valid-looking wrong answer.
Swap the ordering convention instead
($\lambda_1 = 3$, $\lambda_2 = 8$) and $s = 3 - 16 = -13 \to 26$: not
an option, so *that* slip at least announces itself.
Normal-form items are engineered so plausible wrong work still
*decodes*; verify the quadratic and the decode, never just the letter.

In [ ]:
# Verify the worked MC end to end.
Sx = np.array([[7., 4.], [1., 4.]])
vals = np.linalg.eig(Sx)[0]
print("imag check:", np.abs(vals.imag).max())
vals = np.sort(vals.real)[::-1]
print("vals desc:", vals)
s = vals[0] - 2 * vals[1]
m = abs(int(round(s)))
decoded = 2 * m + 1 if s >= 0 else 2 * m
print("s =", int(round(s)), " m =", m, " decoded =", decoded, "-> option B")

### Checkpoint 8

1. Same register, new matrix:
   $T = \begin{pmatrix} 5 & 2 \\ 3 & 4 \end{pmatrix}$, eigenvalues
   descending, $s = \lambda_1 - 2\lambda_2$, same decode rule
   ($2m+1$ if $s \ge 0$, else $2m$).
   Work it end to end.
2. Why do the two branches of this decode rule produce *disjoint* value
   sets (odd vs even), and why does an item author care?

## 9. Common Pitfalls I

**Pitfall 1: trusting `eig`'s ordering.**
Nothing about `eig`'s output order is guaranteed — do not assume
descending, ascending, or stability across matrices.
The matrix below comes back with its *smaller* eigenvalue first:

In [ ]:
P1 = np.array([[1., 3.], [2., 2.]])
vals, vecs = np.linalg.eig(P1)
print("raw order:", vals.real)
order = np.argsort(vals.real)[::-1]
print("after the argsort reorder:", vals.real[order])

Fix: reorder *every time* —
`order = np.argsort(vals)[::-1]; vals, vecs = vals[order], vecs[:, order]` —
and remember the columns of `vecs` must be reordered with the **same**
`order`, or your pairs no longer match.

**Pitfall 2: rows-for-columns.**
`vecs[i]` is a row — a meaningless slice across all eigenvectors.
The eigen-equation residual exposes it instantly:

In [ ]:
S = np.array([[4., 2.], [1., 3.]])
vals, vecs = np.linalg.eig(S)
vals, vecs = vals.real, vecs.real

right = vecs[:, 0]   # column: a real eigenvector
wrong = vecs[0]      # row: NOT an eigenvector
print("column residual:", np.abs(S @ right - vals[0] * right).max())
print("row residual   :", np.abs(S @ wrong - vals[0] * wrong).max())

Fix: `vecs[:, i]`, always — and run the residual check as a habit, not
a debugging move.

**Pitfall 3: complex-dtype panic (or complex-dtype neglect).**
`eig` may return complex arrays even when every eigenvalue is real
(imaginary parts $0$).
Neither panic (the values are fine — check `.imag`, keep `.real`) nor
neglect (feeding complex arrays onward makes prints and comparisons
noisy).
And when the imaginary parts are *not* zero — as for the quarter-turn —
that is the honest answer: no real eigendirections exist.

**Pitfall 4: "the" eigenvector.**
Your hand answer $(2, 1)$, a friend's $(-2, -1)$, and `eig`'s
$(0.894, 0.447)$ are all the same eigenvector — direction is the
content, scale (including sign) is convention.
Compare candidate eigenvectors with the proportionality test or the
residual check, never entrywise equality.
Session 2 pins the course's sign-fixing convention for when routes must
be compared entrywise.

### Checkpoint 9

1. A teammate's code sorts `vals` descending but forgets to reorder
   `vecs`.
   Which check from this session catches it, and what will the check
   report?
2. `np.linalg.eig` on a symmetric matrix returns
   `vals = array([2.+0.j, 5.+0.j])`.
   Give the two-move fix that makes downstream code clean and correctly
   ordered.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $A\,(1,1) = (2 + 1,\; 1 + 2) = (3, 3) = 3\,(1,1)$: stretch $3$.
   $A\,(-1,1) = (-2 + 1,\; -1 + 2) = (-1, 1) = 1 \cdot (-1,1)$:
   stretch $1$.
2. Yes: $A\,(2,2) = (6, 6) = 3\,(2,2)$ — being unturned is a property
   of the *direction*, so every nonzero vector along it behaves the
   same way.
3. No: its row shows $\cos(x, Ax) \approx +0.894 \ne 1$, so the output
   is rotated off the input direction.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $(4, 2)$: $S\,(4,2) = (16 + 4,\; 4 + 6) = (20, 10) = 5\,(4,2)$ —
   eigenvector, $\lambda = 5$ (it is $2 \cdot (2,1)$: the same
   direction as the worked check, illustrating scaling freedom).
   $(1, 1)$: $S\,(1,1) = (6, 4)$; cross-product
   $6 \cdot 1 - 4 \cdot 1 = 2 \ne 0$ — not an eigenvector.
2. $S(-q) = -Sq = -\lambda q = \lambda(-q)$.
3. The definition requires $q \ne 0$ precisely to kill this argument:
   $S\,0 = \lambda\,0$ carries no information about $S$, so the zero
   vector is excluded and $7$ need not be an eigenvalue.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $(3-\lambda)(2-\lambda) - 1 \cdot 2 = \lambda^2 - 5\lambda + 4
   = (\lambda - 4)(\lambda - 1)$: eigenvalues $4, 1$ (descending).
2. $(2-\lambda)(5-\lambda) - 3 \cdot 0 = 0$ directly gives
   $\lambda = 2, 5$ — descending: $5, 2$.
   The zero in the bottom-left corner kills the $bc$ term, so the
   diagonal entries *are* the eigenvalues.
3. Without $q \ne 0$, "some $q$ solves $(S - \lambda I)q = 0$" is true
   for every $\lambda$ (take $q = 0$), so the dependent-rows step —
   which characterizes when a *nonzero* solution exists — would be
   bypassed and every number would qualify as an eigenvalue.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. With $S = \begin{pmatrix} 3 & 1 \\ 2 & 2 \end{pmatrix}$:
   $\lambda = 4$: $S - 4I = \begin{pmatrix} -1 & 1 \\ 2 & -2
   \end{pmatrix}$, top row $(-1, 1)$ → $q = (1, 1)$;
   check $S\,(1,1) = (4, 4) = 4\,(1,1)$. ✓
   $\lambda = 1$: $S - I = \begin{pmatrix} 2 & 1 \\ 2 & 1
   \end{pmatrix}$, top row $(2, 1)$ → $q = (1, -2)$;
   check $S\,(1,-2) = (3 - 2,\; 2 - 4) = (1, -2) = 1 \cdot (1,-2)$. ✓
2. The zero row is perpendicular to *everything* — it constrains
   nothing, so it cannot pin down the eigen-direction.
   The rows of $S - \lambda I$ are proportional (that is what made
   $\lambda$ an eigenvalue), so any nonzero row encodes the same
   perpendicularity condition and yields the same direction.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Same eigenvalues $\{4, 1\}$ and the same two directions as your hand
   answers — but you had to (i) reorder the output descending
   (`argsort` … `[::-1]`) and (ii) recognize the unit-scaled columns as
   your hand vectors: $(1,1)/\sqrt2 \approx (0.707, 0.707)$ and
   $(1,-2)/\sqrt5 \approx (0.447, -0.894)$, possibly with flipped
   signs.
2. It grabbed the first *entry of every eigenvector* (a row).
   The eigen-equation residual `max|S q - lam q|` is machine-precision
   for a true eigenvector and order-$1$ for the row slice.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $D^{10}(1,1) = (3^{10} \cdot 1,\; 0.5^{10} \cdot 1)
   = (59049,\; \tfrac{1}{1024})$.
   Direction: essentially the $x$-axis — the $\lambda = 3$
   eigen-direction has crushed the $\lambda = 0.5$ one.
2. Rank 1 ($\lambda = 0$ means the map flattens one direction: rows
   dependent).
   Inputs along that eigen-direction are sent to the zero vector — an
   F3 "collision", and the reason no undo exists.
3. The direction settles toward the $\lambda = -3$ eigen-direction
   (larger magnitude wins: $|-3| > |1|$), but the iterate's sign along
   it flips on every application.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Entry $(i, j)$ of $Q^{\mathsf T} Q$ is the dot product of columns
   $i$ and $j$ of $Q$ (the Gram matrix of the columns).
   $Q^{\mathsf T}Q = I$ says: diagonal dots (self with self) all $1$ —
   unit length — and off-diagonal dots all $0$ — mutually
   perpendicular.
2. No — $R^{\mathsf T} \ne R$ (the off-diagonal entries are $-1$ and
   $1$), so the promises simply do not apply, and complex eigenvalues
   are permitted.
3. No contradiction: promise 1 says *real*, and negative reals are
   real.
   Geometrically a negative eigenvalue flips its eigen-direction
   (through the origin) while stretching by $|\lambda|$.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. $(5-\lambda)(4-\lambda) - 2 \cdot 3 = \lambda^2 - 9\lambda + 14
   = (\lambda - 7)(\lambda - 2)$: $\lambda_1 = 7$, $\lambda_2 = 2$;
   $s = 7 - 4 = 3 \ge 0$, $m = 3$, decode $2 \cdot 3 + 1 = 7$.
2. Branch $s \ge 0$ emits odd numbers, branch $s < 0$ emits even
   numbers — no value can arise from both branches, so the reported
   number determines $(|s|, \text{sign})$ uniquely and the item stays
   gradable; overlapping branches would let two different computations
   land on the same option.

</details>

<details><summary><b>Checkpoint 9</b></summary>

1. The eigen-equation residual: pairing sorted values with unsorted
   columns mismatches at least two pairs, so
   `max|S q_i - lam_i q_i|` jumps from $\sim 10^{-16}$ to order $1$.
2. ```python
   vals, vecs = np.linalg.eig(S)
   vals, vecs = vals.real, vecs.real      # after checking vals.imag ~ 0
   order = np.argsort(vals)[::-1]
   vals, vecs = vals[order], vecs[:, order]
   ```
   (Two conceptual moves: drop the zero imaginary parts, then reorder
   values and columns *together*.)

</details>